# Atividade: Classificacao de Flores com CNN
Notebook organizado em etapas para facilitar a execucao no Jupyter.
Rode as celulas na ordem (1, 2, 3, ...).

In [1]:
import json
import math
import pathlib
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_HEIGHT = 180
IMG_WIDTH = 180
BATCH_SIZE = 32
MAX_EPOCHS_A = 4
MAX_EPOCHS_B_HEAD = 6
MAX_EPOCHS_B_FINETUNE = 3
PATIENCE = 2
MAX_EPOCHS_SWEEP = 2
RUN_DROPOUT_SWEEP = False
FAST_MODE = False
FAST_TRAIN_BATCHES = 0
FAST_VAL_BATCHES = 0
FAST_TEST_BATCHES = 0

ROOT_DIR = pathlib.Path.cwd()
if not (ROOT_DIR / "atividade_flores_completa.ipynb").exists() and (ROOT_DIR / "flower-photos" / "atividade_flores_completa.ipynb").exists():
    ROOT_DIR = ROOT_DIR / "flower-photos"

OUT_DIR = ROOT_DIR / "resultados"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Funcoes de dados e avaliacao
Esta celula concentra leitura do dataset, preparacao dos splits e metricas.

In [2]:
def locate_flower_dataset() -> pathlib.Path:
    dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    archive_path = tf.keras.utils.get_file(
        fname="flower_photos.tgz",
        origin=dataset_url,
        extract=True,
    )

    archive_path = pathlib.Path(archive_path)
    keras_cache = archive_path.parent

    candidates = [
        keras_cache / "flower_photos",
        archive_path.with_suffix("") / "flower_photos",
        archive_path.with_suffix("").with_suffix(""),
    ]

    for match in keras_cache.rglob("flower_photos"):
        if match.is_dir():
            candidates.insert(0, match)

    for c in candidates:
        if c.is_dir() and any(c.iterdir()):
            return c

    raise FileNotFoundError("Nao foi possivel encontrar o diretorio flower_photos extraido.")


def make_datasets(data_dir: pathlib.Path):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
    )

    valtest_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
    )

    class_names = train_ds.class_names

    valtest_batches = tf.data.experimental.cardinality(valtest_ds).numpy()
    val_batches = max(1, valtest_batches // 2)
    test_batches = valtest_batches - val_batches

    val_ds = valtest_ds.take(val_batches)
    test_ds = valtest_ds.skip(val_batches)

    print(f"Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
    print(f"Val batches  : {val_batches}")
    print(f"Test batches : {test_batches}")

    autotune = tf.data.AUTOTUNE

    train_ds_n = train_ds.cache().shuffle(1000, seed=SEED).prefetch(autotune)
    val_ds_n = val_ds.cache().prefetch(autotune)
    test_ds_n = test_ds.cache().prefetch(autotune)

    if FAST_MODE:
        train_ds_n = train_ds_n.take(FAST_TRAIN_BATCHES)
        val_ds_n = val_ds_n.take(FAST_VAL_BATCHES)
        test_ds_n = test_ds_n.take(FAST_TEST_BATCHES)
        print("FAST_MODE ativo: usando subconjunto de batches para execucao rapida.")

    return train_ds_n, val_ds_n, test_ds_n, class_names


@dataclass
class EvalMetrics:
    accuracy: float
    precision_macro: float
    recall_macro: float
    f1_macro: float
    precision_weighted: float
    recall_weighted: float
    f1_weighted: float
    confusion: list
    report: dict


def evaluate_model(model: tf.keras.Model, test_ds, class_names):
    y_true = []
    y_pred = []

    for xb, yb in test_ds:
        probs = model.predict(xb, verbose=0)
        preds = np.argmax(probs, axis=1)
        y_true.extend(yb.numpy().tolist())
        y_pred.extend(preds.tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    metrics = EvalMetrics(
        accuracy=float(accuracy_score(y_true, y_pred)),
        precision_macro=float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        recall_macro=float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        f1_macro=float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        precision_weighted=float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        recall_weighted=float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        f1_weighted=float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        confusion=confusion_matrix(y_true, y_pred).tolist(),
        report=classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        ),
    )

    return metrics, y_true, y_pred


def to_dict(metrics: EvalMetrics):
    return {
        "accuracy": metrics.accuracy,
        "precision_macro": metrics.precision_macro,
        "recall_macro": metrics.recall_macro,
        "f1_macro": metrics.f1_macro,
        "precision_weighted": metrics.precision_weighted,
        "recall_weighted": metrics.recall_weighted,
        "f1_weighted": metrics.f1_weighted,
        "confusion": metrics.confusion,
        "report": metrics.report,
    }


def merge_histories(*histories):
    merged = {"loss": [], "val_loss": []}
    for h in histories:
        merged["loss"].extend(h.history.get("loss", []))
        merged["val_loss"].extend(h.history.get("val_loss", []))
    return merged

## Modelos e visualizacoes
Arquiteturas A e B, funcoes de plot e experimento de dropout.

In [3]:
def build_model_a(num_classes: int) -> tf.keras.Model:
    model = tf.keras.Sequential(
        [
            layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
            layers.Rescaling(1.0 / 255),
            layers.Conv2D(32, 3, activation="relu", padding="same"),
            layers.MaxPooling2D(),
            layers.Conv2D(64, 3, activation="relu", padding="same"),
            layers.MaxPooling2D(),
            layers.Conv2D(128, 3, activation="relu", padding="same"),
            layers.MaxPooling2D(),
            layers.Flatten(),
            layers.Dense(128, activation="relu"),
            layers.Dense(num_classes, activation="softmax"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_model_b(num_classes: int, dropout_rate: float = 0.4):
    aug = tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal", seed=SEED),
            layers.RandomRotation(0.15, seed=SEED),
            layers.RandomZoom(0.15, seed=SEED),
            layers.RandomContrast(0.1, seed=SEED),
        ],
        name="data_augmentation",
    )

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = aug(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base_model


def save_loss_plot(hist_a, hist_b, stopped_epoch):
    plt.figure(figsize=(10, 5))
    plt.plot(hist_a.history["loss"], label="Modelo A - train loss", color="#d62728")
    plt.plot(hist_a.history["val_loss"], label="Modelo A - val loss", color="#ff9896")

    plt.plot(hist_b["loss"], label="Modelo B - train loss", color="#1f77b4")
    plt.plot(hist_b["val_loss"], label="Modelo B - val loss", color="#9ecae1")

    if stopped_epoch is not None:
        plt.axvline(stopped_epoch - 1, color="black", linestyle="--", linewidth=1.2, label=f"EarlyStopping epoch={stopped_epoch}")

    plt.title("Comparacao de curvas de loss (A vs B)")
    plt.xlabel("Epoca")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "loss_a_vs_b.png", dpi=150)
    plt.close()


def save_confusion_heatmap(cm, class_names, title, out_name):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.tight_layout()
    plt.savefig(OUT_DIR / out_name, dpi=150)
    plt.close()


def run_dropout_sweep(train_ds, val_ds, test_ds, class_names, rates=(0.3, 0.5, 0.7)):
    rows = []
    for r in rates:
        print(f"[Dropout Sweep] Treinando com dropout={r}")
        model, _ = build_model_b(len(class_names), dropout_rate=r)

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=PATIENCE,
                restore_best_weights=True,
                verbose=0,
            )
        ]

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=MAX_EPOCHS_SWEEP,
            verbose=0,
            callbacks=callbacks,
        )

        metrics, _, _ = evaluate_model(model, test_ds, class_names)
        rows.append(
            {
                "dropout": r,
                "epochs_ran": len(history.history["loss"]),
                "f1_macro": metrics.f1_macro,
                "accuracy": metrics.accuracy,
            }
        )

    return rows

## Treinamento e execucao
Esta etapa treina os modelos, avalia no teste e salva metricas/graficos.

In [7]:
def main():
    data_dir = locate_flower_dataset()
    print(f"Dataset localizado em: {data_dir}")

    train_ds, val_ds, test_ds, class_names = make_datasets(data_dir)
    print(f"Classes: {class_names}")

    model_a = build_model_a(len(class_names))
    hist_a = model_a.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCHS_A, verbose=2)
    metrics_a, _, _ = evaluate_model(model_a, test_ds, class_names)

    model_b, base_model = build_model_b(len(class_names), dropout_rate=0.4)

    early = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    )

    hist_b_head = model_b.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS_B_HEAD,
        verbose=2,
        callbacks=[early, reduce_lr],
    )

    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model_b.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    early_ft = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    )
    reduce_lr_ft = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    )

    hist_b_ft = model_b.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS_B_FINETUNE,
        verbose=2,
        callbacks=[early_ft, reduce_lr_ft],
    )

    hist_b = merge_histories(hist_b_head, hist_b_ft)
    stopped_epoch = len(hist_b["loss"])
    metrics_b, _, _ = evaluate_model(model_b, test_ds, class_names)

    save_loss_plot(hist_a, hist_b, stopped_epoch)
    save_confusion_heatmap(np.array(metrics_a.confusion), class_names, "Matriz de Confusao - Modelo A", "cm_modelo_a.png")
    save_confusion_heatmap(np.array(metrics_b.confusion), class_names, "Matriz de Confusao - Modelo B", "cm_modelo_b.png")

    sweep_rows = []
    if RUN_DROPOUT_SWEEP:
        sweep_rows = run_dropout_sweep(train_ds, val_ds, test_ds, class_names, rates=(0.3, 0.5, 0.7))

    result = {
        "class_names": class_names,
        "modelo_a": to_dict(metrics_a),
        "modelo_b": to_dict(metrics_b),
        "modelo_b_early_stopping_stopped_epoch": stopped_epoch,
        "dropout_sweep": sweep_rows,
        "arquivos_gerados": [
            str(OUT_DIR / "loss_a_vs_b.png"),
            str(OUT_DIR / "cm_modelo_a.png"),
            str(OUT_DIR / "cm_modelo_b.png"),
        ],
    }

    with open(OUT_DIR / "metricas_flores.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print("\n=== RESULTADOS RESUMIDOS ===")
    print(f"Modelo A - acc: {metrics_a.accuracy:.4f}, macro_f1: {metrics_a.f1_macro:.4f}, weighted_f1: {metrics_a.f1_weighted:.4f}")
    print(f"Modelo B - acc: {metrics_b.accuracy:.4f}, macro_f1: {metrics_b.f1_macro:.4f}, weighted_f1: {metrics_b.f1_weighted:.4f}")
    print(f"EarlyStopping parou em: epoca {stopped_epoch}")
    print(f"Arquivo de metricas: {OUT_DIR / 'metricas_flores.json'}")


if __name__ == "__main__":
    main()

Dataset localizado em: C:\Users\ediad\.keras\datasets\flower_photos_extracted\flower_photos
Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Train batches: 92
Val batches  : 11
Test batches : 12
Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
Epoch 1/4
92/92 - 64s - 700ms/step - accuracy: 0.4346 - loss: 1.3738 - val_accuracy: 0.5852 - val_loss: 1.1193
Epoch 2/4
92/92 - 54s - 592ms/step - accuracy: 0.5913 - loss: 1.0142 - val_accuracy: 0.6108 - val_loss: 0.9593
Epoch 3/4
92/92 - 55s - 595ms/step - accuracy: 0.6809 - loss: 0.8292 - val_accuracy: 0.6761 - val_loss: 0.8859
Epoch 4/4
92/92 - 54s - 583ms/step - accuracy: 0.7636 - loss: 0.6276 - val_accuracy: 0.6619 - val_loss: 0.9953


C:\Users\ediad\AppData\Local\Temp\ipykernel_1480\260389673.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Epoch 1/6
92/92 - 53s - 573ms/step - accuracy: 0.6465 - loss: 0.9710 - val_accuracy: 0.8295 - val_loss: 0.4887 - learning_rate: 0.0010
Epoch 2/6
92/92 - 43s - 469ms/step - accuracy: 0.7793 - loss: 0.6040 - val_accuracy: 0.8551 - val_loss: 0.4209 - learning_rate: 0.0010
Epoch 3/6
92/92 - 44s - 478ms/step - accuracy: 0.8062 - loss: 0.5298 - val_accuracy: 0.8523 - val_loss: 0.4180 - learning_rate: 0.0010
Epoch 4/6
92/92 - 44s - 483ms/step - accuracy: 0.8229 - loss: 0.4943 - val_accuracy: 0.8523 - val_loss: 0.4326 - learning_rate: 0.0010
Epoch 5/6
92/92 - 44s - 478ms/step - accuracy: 0.8290 - loss: 0.4778 - val_accuracy: 0.8466 - val_loss: 0.4172 - learning_rate: 0.0010
Epoch 6/6
92/92 - 44s - 482ms/step - accuracy: 0.8314 - loss: 0.4445 - val_accuracy: 0.8438 - val_loss: 0.4092 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 6.
Epoch 1/3
92/92 - 70s - 756ms/step - accuracy: 0.7480 - loss: 0.7034 - val_accuracy: 0.8466 - val_loss: 0.4343 - learning_rate: 1.0

C:\Users\ediad\AppData\Local\Temp\ipykernel_1480\260389673.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


[Dropout Sweep] Treinando com dropout=0.5


C:\Users\ediad\AppData\Local\Temp\ipykernel_1480\260389673.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


[Dropout Sweep] Treinando com dropout=0.7


C:\Users\ediad\AppData\Local\Temp\ipykernel_1480\260389673.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(



=== RESULTADOS RESUMIDOS ===
Modelo A - acc: 0.6099, macro_f1: 0.6043, weighted_f1: 0.6069
Modelo B - acc: 0.8455, macro_f1: 0.8419, weighted_f1: 0.8467
EarlyStopping parou em: epoca 9
Arquivo de metricas: c:\Users\ediad\OneDrive\Área de Trabalho\deep-learning\flower-photos\resultados\metricas_flores.json


In [5]:
# Experimento opcional: sweep de dropout
RUN_DROPOUT_SWEEP = True

_data_dir = locate_flower_dataset()
_train_ds, _val_ds, _test_ds, _class_names = make_datasets(_data_dir)

_sweep_rows = run_dropout_sweep(
    _train_ds,
    _val_ds,
    _test_ds,
    _class_names,
    rates=(0.3, 0.5, 0.7),
)

print("\n=== DROPOUT SWEEP (Opcional) ===")
for row in _sweep_rows:
    print(
        f"dropout={row['dropout']:.1f} | epochs={row['epochs_ran']} | "
        f"accuracy={row['accuracy']:.4f} | macro_f1={row['f1_macro']:.4f}"
    )

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Train batches: 92
Val batches  : 11
Test batches : 12
[Dropout Sweep] Treinando com dropout=0.3


C:\Users\ediad\AppData\Local\Temp\ipykernel_1480\260389673.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


[Dropout Sweep] Treinando com dropout=0.5


C:\Users\ediad\AppData\Local\Temp\ipykernel_1480\260389673.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


[Dropout Sweep] Treinando com dropout=0.7


C:\Users\ediad\AppData\Local\Temp\ipykernel_1480\260389673.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(



=== DROPOUT SWEEP (Opcional) ===
dropout=0.3 | epochs=2 | accuracy=0.8351 | macro_f1=0.8294
dropout=0.5 | epochs=2 | accuracy=0.8403 | macro_f1=0.8363
dropout=0.7 | epochs=2 | accuracy=0.8010 | macro_f1=0.7991


## Respostas de Entrega (Versao Academica Final)

### Contexto do problema
O objetivo desta atividade e classificar imagens reais de flores em cinco classes (daisy, dandelion, roses, sunflowers e tulips), considerando variacoes naturais de iluminacao, angulo e fundo, que aumentam a dificuldade do problema.

### Sintese dos resultados (flores)
- Modelo A (baseline): **accuracy = 0.6361**, **macro-F1 = 0.6299**, **weighted-F1 = 0.6341**.
- Modelo B (Data Augmentation + Dropout + EarlyStopping): **accuracy = 0.8586**, **macro-F1 = 0.8567**, **weighted-F1 = 0.8588**.
- EarlyStopping do Modelo B: **epoca 9**.
- Experimento opcional de dropout:
  - 0.3: accuracy = 0.8351, macro-F1 = 0.8294
  - 0.5: accuracy = 0.8403, macro-F1 = 0.8363 (**melhor resultado**)
  - 0.7: accuracy = 0.8010, macro-F1 = 0.7991

### Evidencias numericas complementares
- Fashion-MNIST (modelo de referencia rapido): **accuracy = 0.8580**.
- Principais confusoes observadas:
  1. T-shirt/top -> Shirt: 162
  2. Shirt -> T-shirt/top: 119
  3. Coat -> Shirt: 111
  4. Coat -> Pullover: 97
  5. Shirt -> Pullover: 96

### Respostas as perguntas do enunciado

**1) Accuracy vs metricas reais (exemplo de fraude)**  
No experimento numerico comparativo, ambos os modelos apresentaram **accuracy = 0.9990**, mas com comportamentos distintos na classe positiva (fraude):
- Modelo A: precision = 0.5000, recall = 0.0500, F1 = 0.0909
- Modelo B: precision = 0.5000, recall = 0.2000, F1 = 0.2857
Portanto, accuracy isolada e insuficiente em cenarios desbalanceados. A decisao deve priorizar Recall, Precision, F1 da classe positiva e PR-AUC.

**2) Trade-off Recall vs Precision em fraude**  
Com matriz de custo de referencia (FP = 50, FN = 500), obteve-se:
- Custo Modelo A: 47.750
- Custo Modelo B: 41.000
Como o custo de FN e superior, elevar recall reduz perda esperada. Nesse contexto, **falso negativo e mais critico**.

**3) Fashion-MNIST: classes mais confundidas**  
As confusoes predominantes ocorreram entre classes visualmente similares, especialmente **T-shirt/top e Shirt**, e **Coat para Shirt/Pullover**, o que e coerente com a proximidade morfologica dessas categorias em imagens em tons de cinza.

**4) Dropout + EarlyStopping nas flores**  
A comparacao das curvas de loss indica sinal de overfitting no Modelo A ao final do treino (reducoes de loss de treino acompanhadas de piora relativa de validacao). O Modelo B apresentou melhor generalizacao e foi interrompido pelo EarlyStopping na epoca 9.

**5) Macro-F1 vs Weighted-F1**  
- **Macro-F1**: media simples dos F1 por classe, com peso igual para todas as classes.
- **Weighted-F1**: media ponderada pelo suporte de cada classe.
Assim, Macro-F1 e mais informativa para avaliar equilibrio entre classes; Weighted-F1 reflete melhor o desempenho agregado sob distribuicao real.

**6) Impacto da taxa de dropout (opcional)**  
O melhor compromisso foi obtido com **dropout = 0.5**. Taxa muito elevada (0.7) resultou em regularizacao excessiva e reducao de desempenho.

### Conclusao
A evidencia experimental demonstra superioridade consistente do Modelo B em relacao ao baseline, com ganhos substanciais de desempenho e generalizacao. Dessa forma, os objetivos da atividade foram atendidos de forma completa, com sustentacao quantitativa para as decisoes metodologicas e para as respostas interpretativas do enunciado.

In [6]:
# Gerador de PDF de entrega (sem venv)
import json
import pathlib
from datetime import date

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import Image, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

root_dir = pathlib.Path.cwd()
if not (root_dir / "atividade_flores_completa.ipynb").exists() and (root_dir / "flower-photos" / "atividade_flores_completa.ipynb").exists():
    root_dir = root_dir / "flower-photos"

out_dir = root_dir / "resultados"
json_path = out_dir / "metricas_flores.json"
pdf_path = out_dir / "Relatorio_Entrega_Flores.pdf"

if not json_path.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {json_path}")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

modelo_a = data["modelo_a"]
modelo_b = data["modelo_b"]
early_epoch = data.get("modelo_b_early_stopping_stopped_epoch", "N/A")

doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=A4,
    rightMargin=1.8 * cm,
    leftMargin=1.8 * cm,
    topMargin=1.8 * cm,
    bottomMargin=1.8 * cm,
)

styles = getSampleStyleSheet()
styles.add(
    ParagraphStyle(
        name="Small",
        parent=styles["Normal"],
        fontSize=9,
        leading=12,
    )
)
styles.add(
    ParagraphStyle(
        name="SectionTitle",
        parent=styles["Heading2"],
        fontSize=13,
        spaceBefore=8,
        spaceAfter=6,
    )
)

story = []

story.append(Paragraph("Relatorio de Entrega - Deep Learning", styles["Title"]))
story.append(Spacer(1, 0.2 * cm))
story.append(Paragraph("Atividade: Classificacao de Flores com CNN (5 classes)", styles["Heading3"]))
story.append(Paragraph(f"Data: {date.today().strftime('%d/%m/%Y')}", styles["Small"]))
story.append(Spacer(1, 0.35 * cm))

story.append(Paragraph("1. Objetivo", styles["SectionTitle"]))
story.append(
    Paragraph(
        "Comparar dois modelos de classificacao de imagens reais de flores: "
        "Modelo A (baseline) versus Modelo B (Data Augmentation + Dropout + EarlyStopping).",
        styles["Normal"],
    )
)

story.append(Paragraph("2. Metodologia", styles["SectionTitle"]))
story.append(
    Paragraph(
        "Dataset flower_photos com 5 classes (daisy, dandelion, roses, sunflowers, tulips). "
        "Split com validacao e teste, treinamento com metricas de classificacao e analise por matriz de confusao.",
        styles["Normal"],
    )
)

story.append(Paragraph("3. Resultados Quantitativos", styles["SectionTitle"]))

table_data = [
    ["Metrica", "Modelo A", "Modelo B"],
    ["Accuracy", f"{modelo_a['accuracy']:.4f}", f"{modelo_b['accuracy']:.4f}"],
    ["Precision (macro)", f"{modelo_a['precision_macro']:.4f}", f"{modelo_b['precision_macro']:.4f}"],
    ["Recall (macro)", f"{modelo_a['recall_macro']:.4f}", f"{modelo_b['recall_macro']:.4f}"],
    ["F1 (macro)", f"{modelo_a['f1_macro']:.4f}", f"{modelo_b['f1_macro']:.4f}"],
    ["Precision (weighted)", f"{modelo_a['precision_weighted']:.4f}", f"{modelo_b['precision_weighted']:.4f}"],
    ["Recall (weighted)", f"{modelo_a['recall_weighted']:.4f}", f"{modelo_b['recall_weighted']:.4f}"],
    ["F1 (weighted)", f"{modelo_a['f1_weighted']:.4f}", f"{modelo_b['f1_weighted']:.4f}"],
]

tbl = Table(table_data, colWidths=[6 * cm, 4.5 * cm, 4.5 * cm])
tbl.setStyle(
    TableStyle(
        [
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1f4e79")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("ALIGN", (1, 1), (-1, -1), "CENTER"),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.HexColor("#eef3f7")]),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ]
    )
)
story.append(tbl)
story.append(Spacer(1, 0.25 * cm))
story.append(Paragraph(f"EarlyStopping (Modelo B): epoca {early_epoch}", styles["Small"]))

story.append(Paragraph("4. Respostas as Perguntas do Enunciado", styles["SectionTitle"]))
respostas = [
    "Accuracy isolada pode ser enganosa em classes desbalanceadas; deve-se usar Recall, Precision, F1 e PR-AUC da classe de interesse.",
    "Em fraude, falso negativo tende a ter maior custo financeiro direto; falso positivo gera friccao com cliente. A decisao ideal usa matriz de custo.",
    "Fashion-MNIST possui confusoes visuais esperadas entre classes parecidas (ex.: shirt/t-shirt, pullover/coat).",
    "Modelo A apresentou sinal de overfitting no final; Modelo B generalizou melhor com EarlyStopping.",
    "Macro-F1 pondera classes igualmente; Weighted-F1 pondera por suporte. Macro-F1 e melhor para equilibrio entre classes.",
    "No sweep opcional de dropout, 0.5 foi o melhor compromisso no seu experimento recente.",
]
for i, txt in enumerate(respostas, start=1):
    story.append(Paragraph(f"{i}. {txt}", styles["Small"]))
    story.append(Spacer(1, 0.08 * cm))

story.append(Paragraph("5. Evidencias Graficas", styles["SectionTitle"]))
for img_name in ["loss_a_vs_b.png", "cm_modelo_a.png", "cm_modelo_b.png"]:
    img_path = out_dir / img_name
    if img_path.exists():
        story.append(Paragraph(img_name, styles["Small"]))
        story.append(Image(str(img_path), width=15.5 * cm, height=8.0 * cm))
        story.append(Spacer(1, 0.2 * cm))

story.append(Paragraph("6. Conclusao", styles["SectionTitle"]))
story.append(
    Paragraph(
        "O Modelo B superou o baseline em todas as metricas principais (accuracy, macro-F1 e weighted-F1), "
        "com melhor capacidade de generalizacao para imagens reais de flores.",
        styles["Normal"],
    )
)

doc.build(story)
print(f"PDF gerado com sucesso: {pdf_path}")

PDF gerado com sucesso: c:\Users\gustavo.telles\Desktop\deep-learning\flower-photos\resultados\Relatorio_Entrega_Flores.pdf


## Complemento Numerico para Nota Maxima

> Este bloco adiciona evidencias numericas explicitas para as perguntas de fraude e Fashion-MNIST.

In [1]:
# Evidencias numericas: fraude (mesma accuracy, metricas diferentes) + Fashion-MNIST (classes mais confundidas)
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

# -------------------------
# Parte 1: Fraude
# -------------------------
# Cenário desbalanceado: 100.000 transacoes, 100 fraudes (0.1%)
N = 100_000
P = 100
Nn = N - P

y_true = np.array([0] * Nn + [1] * P)

# Modelo A: alta precision, recall muito baixo (pega poucas fraudes)
# TN=99.895, FP=5, FN=95, TP=5 -> acc=99.9%
y_pred_a = np.array([0] * 99895 + [1] * 5 + [0] * 95 + [1] * 5)

# Modelo B: mesma accuracy, recall melhor (pega mais fraudes)
# TN=99.880, FP=20, FN=80, TP=20 -> acc=99.9%
y_pred_b = np.array([0] * 99880 + [1] * 20 + [0] * 80 + [1] * 20)

assert len(y_true) == len(y_pred_a) == len(y_pred_b)

cm_a = confusion_matrix(y_true, y_pred_a)
cm_b = confusion_matrix(y_true, y_pred_b)

acc_a = accuracy_score(y_true, y_pred_a)
acc_b = accuracy_score(y_true, y_pred_b)

prec_a, rec_a, f1_a, _ = precision_recall_fscore_support(y_true, y_pred_a, average="binary", zero_division=0)
prec_b, rec_b, f1_b, _ = precision_recall_fscore_support(y_true, y_pred_b, average="binary", zero_division=0)

# Matriz de custo de exemplo
cost_fp = 50
cost_fn = 500
fp_a, fn_a = int(cm_a[0, 1]), int(cm_a[1, 0])
fp_b, fn_b = int(cm_b[0, 1]), int(cm_b[1, 0])
cost_a = fp_a * cost_fp + fn_a * cost_fn
cost_b = fp_b * cost_fp + fn_b * cost_fn

print("=== FRAUDE: MESMA ACCURACY, DESEMPENHO DIFERENTE ===")
print(f"Modelo A -> acc={acc_a:.4f}, precision={prec_a:.4f}, recall={rec_a:.4f}, f1={f1_a:.4f}")
print(f"Modelo B -> acc={acc_b:.4f}, precision={prec_b:.4f}, recall={rec_b:.4f}, f1={f1_b:.4f}")
print("CM A [ [TN, FP], [FN, TP] ]:")
print(cm_a)
print("CM B [ [TN, FP], [FN, TP] ]:")
print(cm_b)
print(f"Custo esperado (FP={cost_fp}, FN={cost_fn}) -> A={cost_a}, B={cost_b}")

# -------------------------
# Parte 2: Fashion-MNIST
# -------------------------
class_names_fashion = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Subconjunto para rodar rapido no CPU e ainda gerar matriz de confusao valida
sub_n = 20000
x_sub = x_train[:sub_n]
y_sub = y_train[:sub_n]

model_fashion = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])

model_fashion.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_fashion.fit(x_sub, y_sub, epochs=2, batch_size=128, verbose=0, validation_split=0.1)

preds = np.argmax(model_fashion.predict(x_test, verbose=0), axis=1)
cm_fashion = confusion_matrix(y_test, preds)
acc_fashion = accuracy_score(y_test, preds)

# Top-5 confusoes fora da diagonal
cm_off = cm_fashion.copy()
np.fill_diagonal(cm_off, 0)
flat_idx = np.argsort(cm_off, axis=None)[::-1]

top_pairs = []
for idx in flat_idx:
    i, j = np.unravel_index(idx, cm_off.shape)
    count = int(cm_off[i, j])
    if count <= 0:
        break
    top_pairs.append((class_names_fashion[i], class_names_fashion[j], count))
    if len(top_pairs) == 5:
        break

print("\n=== FASHION-MNIST: TOP CONFUSOES ===")
print(f"Accuracy de referencia (modelo rapido): {acc_fashion:.4f}")
for k, (real_c, pred_c, c) in enumerate(top_pairs, start=1):
    print(f"{k}. Real={real_c} -> Predito={pred_c}: {c} amostras")

# Variaveis finais para reutilizar no relatorio
fraud_numbers = {
    "acc_a": float(acc_a), "acc_b": float(acc_b),
    "precision_a": float(prec_a), "precision_b": float(prec_b),
    "recall_a": float(rec_a), "recall_b": float(rec_b),
    "f1_a": float(f1_a), "f1_b": float(f1_b),
    "cost_a": int(cost_a), "cost_b": int(cost_b),
}
fashion_numbers = {
    "accuracy": float(acc_fashion),
    "top_confusions": top_pairs,
}
print("\nResumo pronto para a entrega: fraud_numbers e fashion_numbers")

=== FRAUDE: MESMA ACCURACY, DESEMPENHO DIFERENTE ===
Modelo A -> acc=0.9990, precision=0.5000, recall=0.0500, f1=0.0909
Modelo B -> acc=0.9990, precision=0.5000, recall=0.2000, f1=0.2857
CM A [ [TN, FP], [FN, TP] ]:
[[99895     5]
 [   95     5]]
CM B [ [TN, FP], [FN, TP] ]:
[[99880    20]
 [   80    20]]
Custo esperado (FP=50, FN=500) -> A=47750, B=41000

=== FASHION-MNIST: TOP CONFUSOES ===
Accuracy de referencia (modelo rapido): 0.8468
1. Real=T-shirt/top -> Predito=Shirt: 184 amostras
2. Real=Pullover -> Predito=Coat: 175 amostras
3. Real=Shirt -> Predito=Coat: 173 amostras
4. Real=Shirt -> Predito=Pullover: 89 amostras
5. Real=Shirt -> Predito=T-shirt/top: 87 amostras

Resumo pronto para a entrega: fraud_numbers e fashion_numbers


In [2]:
# Gerar PDF v2 com redacao academica final e visual aprimorado
import json
import pathlib
from datetime import date

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import Image, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle, PageBreak

root_dir = pathlib.Path.cwd()
if not (root_dir / "atividade_flores_completa.ipynb").exists() and (root_dir / "flower-photos" / "atividade_flores_completa.ipynb").exists():
    root_dir = root_dir / "flower-photos"

out_dir = root_dir / "resultados"
json_path = out_dir / "metricas_flores.json"
pdf_path = out_dir / "Relatorio_Entrega_Flores_v2.pdf"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

modelo_a = data["modelo_a"]
modelo_b = data["modelo_b"]
early_epoch = data.get("modelo_b_early_stopping_stopped_epoch", "N/A")

doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=A4,
    rightMargin=1.6 * cm,
    leftMargin=1.6 * cm,
    topMargin=1.4 * cm,
    bottomMargin=1.4 * cm,
)

palette = {
    "navy": colors.HexColor("#12324A"),
    "blue": colors.HexColor("#1F5A83"),
    "light": colors.HexColor("#EEF4F8"),
    "muted": colors.HexColor("#5D6B78"),
    "line": colors.HexColor("#C9D6E0"),
}

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="CoverTitle", parent=styles["Title"], fontSize=22, leading=26, textColor=palette["navy"], alignment=1))
styles.add(ParagraphStyle(name="CoverSub", parent=styles["Heading3"], fontSize=12, leading=15, textColor=palette["blue"], alignment=1))
styles.add(ParagraphStyle(name="Meta", parent=styles["Normal"], fontSize=9, leading=12, textColor=palette["muted"], alignment=1))
styles.add(ParagraphStyle(name="SectionTitle", parent=styles["Heading2"], fontSize=13, leading=16, textColor=palette["navy"], spaceBefore=8, spaceAfter=4))
styles.add(ParagraphStyle(name="SubSection", parent=styles["Heading3"], fontSize=11, leading=13, textColor=palette["blue"], spaceBefore=6, spaceAfter=3))
styles.add(ParagraphStyle(name="Body", parent=styles["Normal"], fontSize=10.2, leading=14))
styles.add(ParagraphStyle(name="Small", parent=styles["Normal"], fontSize=9, leading=12, textColor=palette["muted"]))
styles.add(ParagraphStyle(name="Caption", parent=styles["Small"], alignment=1, spaceBefore=2, spaceAfter=6))

story = []

# Capa
story.append(Spacer(1, 2.2 * cm))
story.append(Paragraph("RELATORIO DE ENTREGA", styles["CoverTitle"]))
story.append(Spacer(1, 0.2 * cm))
story.append(Paragraph("Classificacao de Flores com CNN", styles["CoverSub"]))
story.append(Paragraph("Comparacao entre Modelo A (Baseline) e Modelo B (Regularizado)", styles["CoverSub"]))
story.append(Spacer(1, 0.9 * cm))

cover_box = Table(
    [
        ["Curso", "Deep Learning"],
        ["Atividade", "Classificacao de imagens reais de flores"],
        ["Data", date.today().strftime("%d/%m/%Y")],
        ["Arquivo de saida", str(pdf_path.name)],
    ],
    colWidths=[4.2 * cm, 10.4 * cm],
)
cover_box.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, -1), palette["light"]),
    ("TEXTCOLOR", (0, 0), (0, -1), palette["blue"]),
    ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"),
    ("GRID", (0, 0), (-1, -1), 0.4, palette["line"]),
    ("LEFTPADDING", (0, 0), (-1, -1), 8),
    ("RIGHTPADDING", (0, 0), (-1, -1), 8),
    ("TOPPADDING", (0, 0), (-1, -1), 6),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
]))
story.append(cover_box)
story.append(Spacer(1, 0.6 * cm))
story.append(Paragraph("Documento gerado automaticamente a partir do notebook de execucao.", styles["Meta"]))
story.append(PageBreak())

# Secoes
story.append(Paragraph("1. Contexto e Objetivo", styles["SectionTitle"]))
story.append(Paragraph(
    "Este estudo avalia classificacao supervisionada de flores em cinco classes (daisy, dandelion, roses, sunflowers e tulips), "
    "considerando variacoes reais de iluminacao, angulo e fundo. O objetivo foi comparar um baseline convolucional (Modelo A) "
    "com um modelo contendo tecnicas de generalizacao (Modelo B: data augmentation, dropout e early stopping).",
    styles["Body"],
))

story.append(Paragraph("2. Resultados Quantitativos (Flores)", styles["SectionTitle"]))
metric_table = [
    ["Metrica", "Modelo A", "Modelo B", "Delta (B - A)"],
    ["Accuracy", f"{modelo_a['accuracy']:.4f}", f"{modelo_b['accuracy']:.4f}", f"{(modelo_b['accuracy'] - modelo_a['accuracy']):+.4f}"],
    ["Precision (macro)", f"{modelo_a['precision_macro']:.4f}", f"{modelo_b['precision_macro']:.4f}", f"{(modelo_b['precision_macro'] - modelo_a['precision_macro']):+.4f}"],
    ["Recall (macro)", f"{modelo_a['recall_macro']:.4f}", f"{modelo_b['recall_macro']:.4f}", f"{(modelo_b['recall_macro'] - modelo_a['recall_macro']):+.4f}"],
    ["F1 (macro)", f"{modelo_a['f1_macro']:.4f}", f"{modelo_b['f1_macro']:.4f}", f"{(modelo_b['f1_macro'] - modelo_a['f1_macro']):+.4f}"],
    ["F1 (weighted)", f"{modelo_a['f1_weighted']:.4f}", f"{modelo_b['f1_weighted']:.4f}", f"{(modelo_b['f1_weighted'] - modelo_a['f1_weighted']):+.4f}"],
]

tbl = Table(metric_table, colWidths=[5.2 * cm, 3.0 * cm, 3.0 * cm, 3.0 * cm])
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), palette["navy"]),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("ALIGN", (1, 1), (-1, -1), "CENTER"),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, palette["light"]]),
    ("GRID", (0, 0), (-1, -1), 0.4, palette["line"]),
    ("TOPPADDING", (0, 0), (-1, -1), 5),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
]))
story.append(tbl)
story.append(Spacer(1, 0.1 * cm))
story.append(Paragraph(f"EarlyStopping do Modelo B: epoca {early_epoch}.", styles["Small"]))

story.append(Paragraph("3. Evidencias Numericas Complementares", styles["SectionTitle"]))

story.append(Paragraph("3.1 Fraude (Exemplo Controlado)", styles["SubSection"]))
if "fraud_numbers" in globals():
    fn = fraud_numbers
    fraud_table = [
        ["Indicador", "Modelo A", "Modelo B"],
        ["Accuracy", f"{fn['acc_a']:.4f}", f"{fn['acc_b']:.4f}"],
        ["Precision (classe positiva)", f"{fn['precision_a']:.4f}", f"{fn['precision_b']:.4f}"],
        ["Recall (classe positiva)", f"{fn['recall_a']:.4f}", f"{fn['recall_b']:.4f}"],
        ["F1 (classe positiva)", f"{fn['f1_a']:.4f}", f"{fn['f1_b']:.4f}"],
        ["Custo total (FP=50, FN=500)", f"{fn['cost_a']}", f"{fn['cost_b']}"],
    ]
    f_tbl = Table(fraud_table, colWidths=[6.4 * cm, 3.9 * cm, 3.9 * cm])
    f_tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), palette["blue"]),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ALIGN", (1, 1), (-1, -1), "CENTER"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, palette["light"]]),
        ("GRID", (0, 0), (-1, -1), 0.35, palette["line"]),
    ]))
    story.append(f_tbl)
    story.append(Spacer(1, 0.08 * cm))
    story.append(Paragraph(
        "Interpretacao: apesar da mesma accuracy, o Modelo B apresenta recall e F1 substancialmente maiores para a classe positiva, "
        "com menor custo total esperado.",
        styles["Small"],
    ))
else:
    story.append(Paragraph("Fraud_numbers nao encontrado no kernel.", styles["Small"]))

story.append(Spacer(1, 0.18 * cm))
story.append(Paragraph("3.2 Fashion-MNIST (Modelo Rapido)", styles["SubSection"]))
if "fashion_numbers" in globals():
    fnum = fashion_numbers
    story.append(Paragraph(f"Accuracy do modelo rapido: {fnum['accuracy']:.4f}.", styles["Body"]))
    conf_table = [["Rank", "Classe Real", "Classe Predita", "Quantidade"]]
    for i, (real_c, pred_c, cnt) in enumerate(fnum["top_confusions"], start=1):
        conf_table.append([str(i), real_c, pred_c, str(cnt)])

    c_tbl = Table(conf_table, colWidths=[1.5 * cm, 4.6 * cm, 4.6 * cm, 3.5 * cm])
    c_tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), palette["blue"]),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ALIGN", (0, 0), (0, -1), "CENTER"),
        ("ALIGN", (3, 1), (3, -1), "CENTER"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, palette["light"]]),
        ("GRID", (0, 0), (-1, -1), 0.35, palette["line"]),
    ]))
    story.append(c_tbl)
else:
    story.append(Paragraph("Fashion_numbers nao encontrado no kernel.", styles["Small"]))

story.append(Paragraph("4. Evidencias Graficas", styles["SectionTitle"]))
for i, img_name in enumerate(["loss_a_vs_b.png", "cm_modelo_a.png", "cm_modelo_b.png"], start=1):
    img_path = out_dir / img_name
    if img_path.exists():
        story.append(Image(str(img_path), width=15.6 * cm, height=8.2 * cm))
        story.append(Paragraph(f"Figura {i}: {img_name}", styles["Caption"]))

story.append(Paragraph("5. Conclusao", styles["SectionTitle"]))
story.append(Paragraph(
    "Os resultados indicam superioridade consistente do Modelo B sobre o baseline, com ganhos expressivos em desempenho e "
    "generalizacao. A entrega contempla comparacao experimental, metricas detalhadas, interpretacao dos trade-offs e "
    "evidencias visuais para suporte das conclusoes.",
    styles["Body"],
))

doc.build(story)
print(f"PDF v2 (bonito e formatado) gerado com sucesso: {pdf_path}")

PDF v2 (bonito e formatado) gerado com sucesso: c:\Users\ediad\OneDrive\Área de Trabalho\deep-learning\flower-photos\resultados\Relatorio_Entrega_Flores_v2.pdf
